# Petal-piston reconstruction from photonic-lantern signals: workbench

The working notebook for PLREC. It combines `trainMLP.ipynb`,
`trainMLP_multiwavelength.ipynb` and `PL_EML.ipynb` into one sequence, as
the starting point for refining them.

**Task.** Estimate the 5 petal pistons of the ANDES DM modal basis (modes
0-4; the sixth petal is the reference) from the light in the output cores of
a photonic lantern simulated with cbeam.

| | Part A: one wavelength | Part B: 41 wavelengths |
|---|---|---|
| pistons | uniform in +-lambda/2 (400 nm at 800 nm, 600 nm at 1200 nm) | uniform in +-8 lambda (6.4 um at 800 nm, 9.6 um at 1200 nm) |
| signals | n_cores camera signals at the design wavelength, unit coupled power | n_cores channel powers x 41 wavelengths across the band, fraction of incident power |
| samples | 10000 (8000 train) | 50000 (40000 train) |
| dataset | `<data>/single*.npz` | `<data>/spectral_41wl.npz` |

At one wavelength a piston is seen only modulo lambda, so Part A stays inside
the unambiguous range. Part B goes far beyond it, and only the combination of
wavelengths can resolve the piston: either one model on the whole spectrum,
or one model per wavelength on the wrapped piston, fused by a
multi-wavelength unwrap.

Each part goes from the simplest reconstructor to the most complex and ends
with a comparison; the summary at the end puts both side by side. The MLP
hyperparameter search of `scripts/mlp_hpo.py` (pl19_800 only) is in
`mlp_hpo.ipynb`. The helper functions are in `ml_reconstruct/notebook.py`.

**Lantern.** The notebook runs for the lantern preset named by the environment
variable `CBEAM_LANTERN` (cbeam's `batch_pipeline.LANTERNS`): `pl19_800`, the
reference 19-port lantern at 800 nm (default); `pl7_1200`, 7 ports at
1200 nm; `pl19_1200`, 19 ports at 1200 nm in the same fibres as `pl7_1200`.
Its settings (piston range, polynomial budgets, data folder) are in
`ml_reconstruct.presets`; executed copies for each lantern are in
`runs/<lantern>/`.

Fitted models are cached under the lantern's `cache_*` folders by name (the
same names as the three single notebooks, so a run loads them instead of
refitting); set `RETRAIN = True` to refit everything. The cache does not check
the data: delete those folders after regenerating a dataset.


In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt
import ml_reconstruct as ml
from ml_reconstruct.notebook import *            # helpers: datasets, scoring, plots, fit-or-load

RETRAIN = False                 # True: refit every model instead of loading the cache
M_MODES = 5                     # DM modes 0..4 (petal pistons)

from ml_reconstruct.presets import settings
S = settings()                  # this lantern's settings (CBEAM_LANTERN)
LANTERN, DATA = S["lantern"], S["data_dir"]
print(f"lantern {LANTERN}, data in {DATA}")

# Part A: one wavelength

Origin: `trainMLP.ipynb`.

## A1. Dataset

Each sample is a random configuration of the 5 pistons, propagated through
the lantern. Each core's signal is the mean of a 5x5-pixel window of the
imaged output face around the core centre, with the input normalised to unit
coupled power (`ml_reconstruct.simulate.simulate_core_images`). If the file
is missing it is simulated: 15-40 minutes on the numpy backend.

**Why the port count matters at one wavelength.** Each single piston is kept
inside +-lambda/2, but two petals can differ by twice that. The signals of an
N-port lantern, normalised, have N - 1 independent values; a smooth map of the
5 pistons is in general one-to-one over a large range only into 11 or more
dimensions, otherwise it folds onto itself and different piston sets give the
same signals. 19 ports (18 values) resolve the range; 7 ports (6 values) do
not: on `pl7_1200` every reconstructor stops far above the 19-port lanterns,
while the same lantern reconstructs smaller pistons well. At exactly +-lambda/2
a piston equals its opposite (+lambda/2 and -lambda/2 give the same field), so
the edges of the range are ambiguous for every lantern.
`pl19_1200` (same fibres and wavelength, 19 ports) is the direct test.

Signals and pistons are standardised with the training-split statistics.


In [ ]:
A_SAMPLES, A_TRAIN = 10000, 8000
A_SPAN_NM = S["single_span_nm"]
A_DATASET = S["single"]
# phasor-target models; cache_single keeps the earlier direct-piston fits
cacheA = ml.ModelCache(f"{DATA}/cache_single_phasor", retrain=RETRAIN)
# simulated (with rng=0) only if the file is missing
dsA = load_or_simulate(A_DATASET, "make_single_dataset", n=A_SAMPLES, n_train=A_TRAIN, span_nm=A_SPAN_NM,
                       lantern=LANTERN)

In [ ]:
SA_tr, CA_tr = dsA.train
SA_va, CA_va = dsA.val
# At one wavelength p and p +- lambda give the same signals: the models regress the phasor
# (sin, cos)(2 pi p / lambda) of each piston, which has no jump at +-lambda/2, decoded with atan2
LAMBDA_A = 2 * A_SPAN_NM     # the range is +-lambda/2 at the design wavelength
xsA, ysA = ml.Standardizer().fit(SA_tr), ml.PhasorCodec(LAMBDA_A)
OPP = ysA.outputs_per_piston  # 2 fitted outputs per piston: degrees of freedom per piston count both
XA_tr, XA_va = xsA.transform(SA_tr), xsA.transform(SA_va)
YA_tr, YA_va = ysA.transform(CA_tr), ysA.transform(CA_va)
print(f"{len(SA_tr)} training / {len(SA_va)} validation samples, {SA_tr.shape[1]} cores, "
      f"{CA_tr.shape[1]} pistons, std {CA_tr.std():.0f} nm")

RESULTS_A, DOF_A = {}, {}   # model -> per-piston validation RMSE [nm], degrees of freedom per piston
WRAP_A = {}                  # the same with the errors modulo lambda: the figure of merit

## A1b. A look at the dataset

The piston distribution, and the core signals across the training set: their
spread per core, their mean and relative variation on the output layout, and
how they correlate with each piston. Correlation with p shows the part of a
core's response that is odd in the piston, and correlation with p^2 the even
part.

In [ ]:
plot_single_overview(SA_tr, CA_tr, dsA.core_xy)

In [ ]:
plot_signal_piston_correlation(SA_tr, CA_tr)
tot = SA_tr.sum(axis=1)
print(f"sum of the {SA_tr.shape[1]} signals per sample: mean {tot.mean():.4g}, std / mean {tot.std() / tot.mean():.2%}")

## A2. Baseline: k-nearest neighbours

k-NN needs no training and shows how well the signals determine the pistons
at all.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knnA = cacheA("kNN", lambda: KNeighborsRegressor(5, weights="distance").fit(XA_tr, YA_tr))
score(RESULTS_A, "kNN", knnA.predict(XA_va), CA_va, ys=ysA, period=LAMBDA_A, wrapped=WRAP_A)
DOF_A["kNN"] = n_dof(knnA, OPP)          # k-NN fits nothing: the training values it keeps, per piston

## A3. MLP

Three hidden layers of 128 units, Adam, 500 epochs; the weights of the epoch
with the lowest validation loss are kept.

In [ ]:
LAYERS_A = [XA_tr.shape[1], 128, 128, 128, YA_tr.shape[1]]     # (sin, cos) of each piston
MLP_A = f"{DATA}/cache_single_phasor/mlp-{'x'.join(map(str, LAYERS_A))}-e500.npz"
paramsA, histA = fit_or_load_mlp(MLP_A, LAYERS_A, XA_tr, YA_tr, XA_va, YA_va, epochs=500, lr=3e-4,
                                 retrain=RETRAIN)
CA_mlp = ysA.inverse(np.asarray(ml.mlp_forward(paramsA, XA_va)))
score(RESULTS_A, "MLP", CA_mlp, CA_va, period=LAMBDA_A, wrapped=WRAP_A)
DOF_A["MLP"] = n_dof(paramsA, OPP)
plot_mlp(histA, CA_va, CA_mlp, k=2, span=A_SPAN_NM)

## A4. Polynomial models


### Zernike representation of the core signals

The core signals, placed at their cores on the unit disk, projected onto the
first `n_zern` Noll Zernikes (36 for 19 cores; 10 for 7 cores, the fewest with
full rank on one hexagonal ring, where coma coincides with tilt). The sampled
basis has rank n_cores, so this is a change of variables rather than a
compression, shown as a diagnostic of how the mean core signal decomposes.


In [ ]:
N_ZERN = S["n_zern"]
ZBASIS = ml.zernike_basis(dsA.core_xy, N_ZERN, fill=0.8)[0]
ZPROJ = ml.modal_projector(ZBASIS)
print(f"Zernike basis {ZBASIS.shape}, rank {np.linalg.matrix_rank(ZBASIS)}")

# diagnostic only
s_mean = SA_tr.mean(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
hexmap(ax[0], dsA.core_xy, s_mean, "mean core signal")
hexmap(ax[1], dsA.core_xy, ZBASIS @ (ZPROJ @ s_mean), "Zernike reconstruction")
plt.tight_layout(); plt.show()


### Linear and dense polynomial ridge (degree 2 and 3) on the core signals

In [ ]:
# Dense ridge on every monomial of degree <= 1, 2, 3 of the core signals (19 / 209 / 1539 terms),
# regressing the phasors; the penalty per output by efficient leave-one-out.
for deg, name in ((1, "linear"), (2, "dense poly, degree 2"), (3, "dense poly, degree 3")):
    model = cacheA(f"fiber-d{deg}", lambda: ml.fit_dense_poly_ridge(XA_tr, YA_tr, deg))
    DOF_A[name] = n_dof(model, OPP)
    score(RESULTS_A, name, ml.predict_dense_poly_ridge(model, XA_va), CA_va, ys=ysA, period=LAMBDA_A,
          wrapped=WRAP_A, note=f"   ({model[0].n_output_features_} terms)")

### Compact model: OMP with a term budget K

Orthogonal Matching Pursuit keeps at most K of the degree-3 monomials of the
standardised core signals per piston (1539 for 19 cores, 119 for 7) (after screening them by correlation
with the piston). The sweep shows the accuracy / size trade-off.

**Choosing the OMP budget K, per lantern.**

- *19 cores* (1539 degree-3 monomials). On `pl19_800` the sweep falls smoothly
  -- 15.5 nm at K = 1100, 9.3 nm at K = 1275 -- and stops at **K = 1281**,
  where every piston keeps its own natural support (1280 or 1281 monomials);
  every larger budget, up to the full cubic basis, reproduces that fit to the
  last digit (8.41 nm). K = 1282 is that saturated model counted with its
  intercept: its degrees of freedom per piston. `pl19_1200` uses the same
  budget; its own sweep shows whether it saturates at the same point.
- *7 cores* (119 monomials). On `pl7_1200` the sweep is nearly flat (from
  ~250 nm at K = 5 to ~224 nm with the complete cubic, K = 119) and there is
  no saturation point to find; the floor is the ambiguity of 7 signals, not
  the model, so K = 119 is used.

*Phasor targets.* The budgets above were chosen when the models regressed p itself. With
the (sin, cos) targets each piston has two OMP fits, each with budget K; noiseless, the
phasor fit at the full cubic budget is exact to a few thousandths of a nm on `pl19_1200`.

In [ ]:
SCREEN_A = 9000
KS_A = S["omp_ks"]
RECORD_KS_A = S["omp_record_wb"]

F3A_tr, F3A_va, _, poly_d3A = ml.poly_features(XA_tr, XA_va, 3)   # core-signal monomials
omp_fits_A = lambda K: omp_fits(cacheA, "fiber-omp", F3A_tr, YA_tr, K, SCREEN_A)
sweepA = omp_sweep(cacheA, "fiber-omp", F3A_tr, YA_tr, F3A_va, CA_va, ysA, KS_A, SCREEN_A,
                   period=LAMBDA_A)


In [ ]:
for K in RECORD_KS_A:
    score(RESULTS_A, f"sparse poly (OMP), K={K}", omp_predict_all(F3A_va, omp_fits_A(K)), CA_va, ys=ysA,
          period=LAMBDA_A, wrapped=WRAP_A)
    DOF_A[f"sparse poly (OMP), K={K}"] = n_dof(omp_fits_A(K), OPP)
plot_omp_sweep(sweepA, "core-signal degree-3 OMP: accuracy vs. model size")


### Closed-form forward model, inverted with a fixed number of steps

With petal pistons the signals are exact ratios of quadratic forms in the six petal phasors
(`ml_reconstruct.forward`, 619 constants fitted by one linear least-squares solve). Its inversion
uses nothing but the model and has a fixed cost (`forward.invert_fixed`): a **spectral start**
as in phase retrieval (the lifted problem is linear in the 30 pair features; minimum-norm solution,
5 alternating projections onto the rank-1 pair matrices, leading eigenvector), its twin image, the
second eigenvector and 32 fixed Sobol points of the +-lambda/2 cube, then **5 Gauss-Newton steps**
from each; the lowest residual is kept.

In [ ]:
from ml_reconstruct import forward as fw, petals
from ml_reconstruct.simulate import bp
_p = bp.get_simulation_parameters(lantern=LANTERN)
_f = bp.load_ifunc(_p["ifunc_file"]); bp.center_ifunc_mask(_f)
_g = petals.petal_geometry(np.asarray(_f.mask_inf_func) > 0, np.asarray(_f.influence_function), 5)
TWIN = petals.twin_matrix(_g["opposite"], _g["ref"])

fwd = cacheA("closed-form", lambda: fw.fit_forward(CA_tr, SA_tr, LAMBDA_A))
CA_cf, _ = fw.invert_fixed(fwd, SA_va, TWIN)
score(RESULTS_A, "closed form", CA_cf, CA_va, period=LAMBDA_A, wrapped=WRAP_A,
      note=f"   ({fwd.n_constants} constants, 35 starts x 5 steps)")
DOF_A["closed form"] = fwd.n_constants / M_MODES          # shared by the 5 pistons

### Symbolic regression

The best symbolic reconstructor found by `scripts/symbolic_sr.py` (Operon and PySR, one search
per class of piston for the sine and the cosine of its phase, on angular-harmonic features of
the core signals): per class the expression pair with the lowest held-out error, from the engine
with the lower mean held-out error. Its validation rows are this notebook's.

In [ ]:
from ml_reconstruct import symbolic as sy
SR = sy.sr_reconstructor(f"{DATA}/symbolic", LAMBDA_A)
if SR is None:
    print("no symbolic-regression results in", f"{DATA}/symbolic")
else:
    sr_engine, CA_sr, sr_const, sr_rows = SR
    score(RESULTS_A, "symbolic regression", CA_sr, CA_va, period=LAMBDA_A, wrapped=WRAP_A,
          note=f"   ({sr_engine}; constants per piston {sr_const.astype(int)})")
    DOF_A["symbolic regression"] = float(np.mean(sr_const))
    for c, r in sr_rows.items():
        print(f"  {c:11s} complexity {r.complexity:3d}: sin = {r.sin_expr}\n{'':29s}cos = {r.cos_expr}")
    # accuracy vs. complexity of the chosen engine, per class (validation RMSE modulo lambda)
    res = sy.load_sr(f"{DATA}/symbolic", sr_engine)
    fronts = {f"{sr_engine}, {c}": sy.sr_front(res, c, LAMBDA_A) for c in sr_rows}
    sy.plot_sr_fronts(fronts, {"linear": WRAP_A["linear"].mean(),
                               "dense poly, degree 2": WRAP_A["dense poly, degree 2"].mean()})

## A5. What the OMP models use

### Which monomials does OMP keep?

Kept vs. available monomials per degree at K = `k_table` (500 for 19 cores, 60
for 7), and how the degree mix of
the kept terms changes with the budget.

In [ ]:
K_TABLE, KS_MIX = S["k_table"], S["ks_mix"]
degree = poly_d3A.powers_.sum(axis=1)
degs = np.arange(1, degree.max() + 1)
total = np.array([(degree == d).sum() for d in degs])
kept = kept_by_degree(omp_fits_A(K_TABLE), degree)
print(f"K = {K_TABLE}: kept / available monomials per degree (one column per fit: sin of pistons 0-4, then cos)")
for i, d in enumerate(degs):
    print(f"  degree {d}: {total[i]:5d} available, kept {kept[:, i]}  ({kept[:, i].mean() / total[i]:.1%})")

In [ ]:
mix = np.array([kept_by_degree(omp_fits_A(K), degree).sum(axis=0) for K in KS_MIX], float)
plot_degree_mix(KS_MIX, (mix / mix.sum(axis=1, keepdims=True)).T)

### Which core signals matter?

For the largest recorded OMP model: the total |coefficient| (in standardised units) of
the monomials each core signal appears in, per piston (core 0 is the central
one).

In [ ]:
K = S["k_importance"]
mass = omp_variable_importance(omp_fits_A(K), poly_d3A.powers_)   # (fit, core signal)
mass = mass[:M_MODES] + mass[M_MODES:]                            # sin + cos fits of each piston
vnames = [f"core {i}" for i in range(XA_tr.shape[1])]
plot_stacked_importance(mass, vnames, f"core-signal importance in fiber-omp-K{K}")


## A6. Part A comparison

Every model regresses the phasor of each piston and is decoded into [-lambda/2, lambda/2).
Plain RMSE first, then the RMSE of the errors taken modulo lambda (an estimate across the
wrap is the right piston): the figure of merit. Noiseless, the cubic models are essentially
exact; the photon-noise comparison is at the end of `trainMLP.ipynb`.

In [ ]:
HIDE_A = ("kNN", "sparse poly (OMP), K=500")       # kept in the tables, not in the plots
show_results(RESULTS_A, f"Part A ({LANTERN}): one wavelength, +-{A_SPAN_NM:.0f} nm, by decreasing mean RMSE",
             logy=False, sort=True, dof=DOF_A, exclude=HIDE_A)
show_results(WRAP_A, f"Part A ({LANTERN}): one wavelength, +-{A_SPAN_NM:.0f} nm, errors modulo lambda",
             logy=True, sort=True, dof=DOF_A, exclude=HIDE_A, ylabel="validation RMSE modulo lambda (nm)")

# Part B: 41 wavelengths, +-8 lambda

Origin: `trainMLP_multiwavelength.ipynb` (single models on the whole
spectrum) and `PL_EML.ipynb` (per-wavelength models and the unwrap).

## B1. Dataset

One lantern characterization per wavelength (cached in cbeam's
`examples/data/`), then all samples propagated wavelength by wavelength
(`ml_reconstruct.simulate.simulate_spectra`). Each signal is the power in
one output channel as a fraction of the incident power, so it includes the
input coupling efficiency. Simulating 50000 samples at 41 wavelengths in one
process takes many hours; `gen_datasets.py` / `run_gen.sh` split it into 410
independent pieces (41 wavelengths x 10 chunks) run in parallel, which is how
the current file was made.

The signals are standardised per (wavelength, core) column with the
training statistics. `XB_*` is the flat (41 x n_cores)-long view used by the
full-spectrum models, `X3B_*` the same numbers as (n, wavelength, core) for
the per-wavelength models.

In [ ]:
B_SAMPLES, B_TRAIN = 50000, 40000
B_SPAN_NM = S["spectral_span_nm"]               # +-8 lambda
from ml_reconstruct.simulate import bp            # cbeam's batch_pipeline
WAVELENGTHS_NM = np.linspace(*bp.get_simulation_parameters(lantern=LANTERN)["band_nm"], 41)
B_DATASET = S["spectral"]
cacheB = ml.ModelCache(f"{DATA}/cache_spectral", retrain=RETRAIN)     # full-spectrum models
cacheW = ml.ModelCache(f"{DATA}/cache_perlambda", retrain=RETRAIN)    # per-wavelength and global OMP
# simulated (with rng=0) only if the file is missing: hours, see gen_datasets.py for a parallel run
dsB = load_or_simulate(B_DATASET, "make_spectral_dataset", n=B_SAMPLES, n_train=B_TRAIN,
                       span_nm=B_SPAN_NM, wavelengths_nm=WAVELENGTHS_NM, lantern=LANTERN)

In [ ]:
SB_tr, CB_tr = dsB.train
SB_va, CB_va = dsB.val
lam = dsB.wavelengths_nm
n_wl, n_cores = SB_tr.shape[1:]
xsB, ysB = ml.Standardizer().fit(SB_tr.reshape(len(SB_tr), -1)), ml.Standardizer().fit(CB_tr)
XB_tr = xsB.transform(SB_tr.reshape(len(SB_tr), -1))
XB_va = xsB.transform(SB_va.reshape(len(SB_va), -1))
X3B_tr, X3B_va = XB_tr.reshape(SB_tr.shape), XB_va.reshape(SB_va.shape)
YB_tr, YB_va = ysB.transform(CB_tr), ysB.transform(CB_va)
print(f"{len(XB_tr)} training / {len(XB_va)} validation samples, {n_wl} wavelengths "
      f"({lam[0]:.0f}-{lam[-1]:.0f} nm) x {n_cores} cores = {XB_tr.shape[1]} inputs; "
      f"pistons std {CB_tr.std():.0f} nm")

In [ ]:
RESULTS_B, DOF_B = {}, {}   # model -> per-piston validation RMSE [nm], degrees of freedom per piston
score(RESULTS_B, "no skill (mean)", np.broadcast_to(CB_tr.mean(0), CB_va.shape), CB_va)
DOF_B["no skill (mean)"] = 1          # one constant per piston

## B1b. A look at the dataset

The rows below cover:

- **Total channel power vs. wavelength.** The total for the random pistons
  of the training set, compared with a flat wavefront when the ramp results
  of `multi_wavelength.ipynb` are available.
- **One sample's spectra.**
- **Mean channel power and relative variation, per (wavelength, core).**
- **How much each wavelength sees each piston, linearly.** This is the R^2 of
  a linear fit of the piston's phasor (sin, cos)(2 pi p / lambda) on the
  channel powers at that wavelength, averaged over sin and cos, on a quarter
  of the training rows. It ignores everything a nonlinear model can add.

In [ ]:
m_tot = plot_total_power(lam, SB_tr, ramp_file=S["ramp"])   # ramp: multi_wavelength.ipynb
print("largest steps of the mean total power between neighbouring wavelengths:")
for j in np.argsort(-np.abs(np.diff(m_tot)))[:3]:
    print(f"  {lam[j]:.0f} -> {lam[j + 1]:.0f} nm: {m_tot[j]:.3f} -> {m_tot[j + 1]:.3f}")

In [ ]:
plot_sample_spectra(lam, SB_tr, CB_tr, i=0)
plot_spectral_maps(lam, SB_tr)

In [ ]:
rows_s = ml.subsample_rows(len(SB_tr), 0.25, seed=2)
R2 = phasor_r2(X3B_tr[rows_s], CB_tr[rows_s], lam)
plot_per_piston(lam, R2, "wavelength (nm)", "R^2 of a linear phasor fit",
                "linear sensitivity of the channel powers to each piston's phasor")

## B2. Single models on the whole spectrum

### k-nearest neighbours

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knnB = cacheB("kNN", lambda: KNeighborsRegressor(5, weights="distance").fit(XB_tr, YB_tr))
score(RESULTS_B, "kNN", knnB.predict(XB_va), CB_va, ys=ysB)
DOF_B["kNN"] = n_dof(knnB)

### Dense degree-2 polynomial ridge on a wavelength subset

Every degree-2 monomial of the core signals at every `WL_STRIDE`-th
wavelength (raw signals, or their per-wavelength Zernike projection with
`REPR = "zern"`), one ridge per piston. The number of terms grows
quadratically with the inputs and the memory linearly with the rows, so the
fit uses a random `DATA_SHARE` of the training and validation rows, and
checks the memory needed first. It is scored on that validation subset only.

In [ ]:
REPR, WL_STRIDE, DEGREE_R = "fiber", 4, 2
DATA_SHARE_R, MEM_BUDGET_GIB = 0.15, 128.0
wl_fit = np.arange(0, n_wl, WL_STRIDE)

proj_R = ml.modal_projector(ml.zernike_basis(dsB.core_xy, S["zern_spectral"], fill=0.8)[0]) if REPR == "zern" else None
to_features = lambda S: spectral_features(S, wl_fit, proj_R)
fsB = ml.Standardizer(eps=1e-9).fit(to_features(SB_tr))
fit_rows = ml.subsample_rows(len(SB_tr), DATA_SHARE_R, seed=0)
val_rows = ml.subsample_rows(len(SB_va), DATA_SHARE_R, seed=1)
F_fit, F_val = fsB.transform(to_features(SB_tr[fit_rows])), fsB.transform(to_features(SB_va[val_rows]))

In [ ]:
n_terms_R = ml.n_poly_terms(F_fit.shape[1], DEGREE_R)
need_gib = (1.5 * len(F_fit) + len(F_val)) * n_terms_R * 8 / 2**30   # RidgeCV needs ~0.5x Z_fit extra
print(f"{len(wl_fit)} wavelengths x {F_fit.shape[1] // len(wl_fit)} {REPR} = {F_fit.shape[1]} inputs "
      f"-> {n_terms_R:,} terms; {len(F_fit)} fit / {len(F_val)} val rows -> ~{need_gib:.1f} GiB")
assert need_gib <= MEM_BUDGET_GIB, "lower DATA_SHARE_R (linear) or raise WL_STRIDE (quadratic)"

ridgeB = cacheB(f"{REPR}-ridge-d{DEGREE_R}-w{len(wl_fit)}-n{len(F_fit)}",
                lambda: ml.fit_dense_poly_ridge(F_fit, YB_tr[fit_rows], DEGREE_R, alphas=np.logspace(-2, 6, 17)))
DOF_B[f"{REPR}-ridge-d{DEGREE_R}"] = n_dof(ridgeB)
score(RESULTS_B, f"{REPR}-ridge-d{DEGREE_R}", ysB.inverse(ml.predict_dense_poly_ridge(ridgeB, F_val)),
      CB_va[val_rows], note=f"   (on {len(val_rows)} validation rows)");

### One global polynomial model (OMP)

A single degree-3 OMP model per piston on the core signals of several
wavelengths at once, regressing the piston directly. The number of terms
grows as (inputs)^3, so only a subset of wavelengths and cores fits in
memory. It is fitted on the same 20% of the training rows as the
per-wavelength models of B3.

In [ ]:
DATA_SHARE_W = 0.2
rows_W = ml.subsample_rows(len(X3B_tr), DATA_SHARE_W, seed=0)   # shared with B3

WL_G, FIB_G, K_G, SCREEN_G = np.arange(0, n_wl, S["wl_stride_global"]), np.arange(n_cores), 1000, 6000
F_tr = X3B_tr[rows_W][:, WL_G][:, :, FIB_G].reshape(len(rows_W), -1)
F_va = X3B_va[:, WL_G][:, :, FIB_G].reshape(len(X3B_va), -1)
print(f"{len(WL_G)} wavelengths x {len(FIB_G)} cores = {F_tr.shape[1]} inputs -> "
      f"{ml.n_poly_terms(F_tr.shape[1], 3):,} degree-3 terms, {len(rows_W)} training rows")
globB = cacheW(f"global-d3-K{K_G}-w{len(WL_G)}-f{len(FIB_G)}-n{len(rows_W)}",
               lambda: ml.fit_omp_poly(F_tr, CB_tr[rows_W], 3, K_G, SCREEN_G))
score(RESULTS_B, "global OMP", ml.predict_omp_poly(globB, F_va), CB_va)
DOF_B["global OMP"] = n_dof(globB)

### MLP on the full spectrum

Four hidden layers of 256 units on all 41 x n_cores inputs, Adam, 10000 epochs
(about 5 minutes on a GPU). Its best epoch comes at or near the end of the
run on the lanterns studied so far, so it may not have finished converging.


In [ ]:
LAYERS_B = [XB_tr.shape[1], 256, 256, 256, 256, M_MODES]
EPOCHS_B, LR_B = 10000, 3e-4
MLP_B = f"{DATA}/cache_spectral/mlp-{'x'.join(map(str, LAYERS_B))}-e{EPOCHS_B}-lr{LR_B:g}.npz"
paramsB, histB = fit_or_load_mlp(MLP_B, LAYERS_B, XB_tr, YB_tr, XB_va, YB_va, epochs=EPOCHS_B, lr=LR_B,
                                 retrain=RETRAIN)
CB_mlp = ysB.inverse(np.asarray(ml.mlp_forward(paramsB, XB_va)))
score(RESULTS_B, "MLP (all wl)", CB_mlp, CB_va)
DOF_B["MLP (all wl)"] = n_dof(paramsB)
plot_mlp(histB, CB_va, CB_mlp, k=2, span=B_SPAN_NM, s=2)

## B3. Per-wavelength models of the wrapped piston

At wavelength lambda the signals depend on a piston only modulo lambda (the
pupil phase is 2 pi p / lambda). So one degree-3 OMP polynomial (at most K
terms) of the core signals per wavelength in `WL_USE` regresses the phasor
(sin, cos)(2 pi p / T) of each piston wrapped at the period T = lambda, on
the training rows `rows_W`.

In [ ]:
WRAP_FACTOR = 1.0                  # period = WRAP_FACTOR * lambda (0.5 also works, same accuracy:
                                   # a function of p mod lambda/2 is one of p mod lambda)
WL_USE = np.arange(0, n_wl, 2)     # 21 of the 41 wavelengths
FIB_USE = np.arange(n_cores)       # all cores
DEGREE_W, K_W, SCREEN_W = 3, S["k_wrap"], 2000    # 19 cores: 1282 of 1539 terms; 7 cores: all 119

periods = lam * WRAP_FACTOR
TAG_W = f"wrap{WRAP_FACTOR:g}-d{DEGREE_W}-K{K_W}-f{len(FIB_USE)}-n{len(rows_W)}"
fitW = ml.fit_wrapped_models(
    X3B_tr, X3B_va, CB_tr, CB_va, periods, WL_USE, FIB_USE, degree=DEGREE_W, K=K_W, screen=SCREEN_W,
    cache=cacheW, rows=rows_W, tag=TAG_W)
print(f"{len(WL_USE)} per-wavelength models ({fitW['seconds']:.0f} s)")

In [ ]:
print_wrapped_table(fitW, lam)
plot_per_piston(lam[WL_USE], fitW["sig_va"], "wavelength (nm)", "wrapped-piston RMSE (nm)",
                "per-wavelength models", logy=True, ref=fitW["periods"] / np.sqrt(12), ref_label="chance")

## B4. Multi-wavelength unwrap

The wrapped estimates of all wavelengths in `WL_USE` are fused into the
full-range piston (`ml_reconstruct.unwrap`: a circular maximum-likelihood
search over the whole +-8 lambda range, then a robust refinement), as in
synthetic-wavelength interferometry. A "wrong fringe" is an error larger than
half the shortest period.

In [ ]:
P_hat, Q = ml.unwrap_all(fitW, B_SPAN_NM)
score(RESULTS_B, "sparse poly (OMP) + unwrap", P_hat, CB_va)
cacheW.verbose = False                 # summed over the per-wavelength models
DOF_B["sparse poly (OMP) + unwrap"] = sum(n_dof(cacheW(f"{TAG_W}-w{w}", None), 2) for w in WL_USE)  # sin + cos
cacheW.verbose = True
print()
print_unwrap_table(P_hat, CB_va, fitW)
plot_unwrap(P_hat, CB_va, Q, B_SPAN_NM)

# The same route with every other single-wavelength method at each wavelength, then the same
# unwrap: dense ridge of degree 1 (linear), 2, 3 on the phasors; a 3 x 128 MLP on the phasors;
# the closed-form model inverted with the fixed-step method (no symbolic regression: one search
# per wavelength and class would take days).
from ml_reconstruct import forward as fw, petals
from ml_reconstruct.simulate import bp
_p = bp.get_simulation_parameters(lantern=LANTERN)
_f = bp.load_ifunc(_p["ifunc_file"]); bp.center_ifunc_mask(_f)
_g = petals.petal_geometry(np.asarray(_f.mask_inf_func) > 0, np.asarray(_f.influence_function), 5)
TWIN = petals.twin_matrix(_g["opposite"], _g["ref"])
FIT_R = {}
for name, kind, deg in (("linear", "ridge", 1), ("dense poly, degree 2", "ridge", 2),
                        ("dense poly, degree 3", "ridge", 3), ("MLP", "mlp", None),
                        ("closed form", "closed_form", None)):
    tag = f"wrap{WRAP_FACTOR:g}-{kind}{'-d%d' % deg if deg else ''}-f{len(FIB_USE)}-n{len(rows_W)}"
    FIT_R[name] = ml.fit_wrapped_models(X3B_tr, X3B_va, CB_tr, CB_va, periods, WL_USE, FIB_USE,
                                        degree=deg or 3, cache=cacheW, rows=rows_W, tag=tag, model=kind,
                                        twin=TWIN)
    P_r, _ = ml.unwrap_all(FIT_R[name], B_SPAN_NM)
    score(RESULTS_B, f"{name} + unwrap", P_r, CB_va)
    cacheW.verbose = False
    per_wl = [cacheW(f"{tag}-w{w}", None) for w in WL_USE]
    DOF_B[f"{name} + unwrap"] = sum((m.n_constants / M_MODES) if kind == "closed_form" else n_dof(m, 2)
                                    for m in per_wl)
    cacheW.verbose = True
    print(f"  wrapped RMSE per wavelength, mean over pistons: "
          f"{FIT_R[name]['sig_va'].mean():.1f} nm; wrong fringes "
          f"{100 * np.mean(np.abs(P_r - CB_va) > FIT_R[name]['periods'].min() / 2):.2f}%")

## B5. Part B comparison

In [ ]:
# the per-wavelength routes (each method + unwrap), sorted by decreasing error; the single models
# on the whole spectrum (kNN, global OMP, full-spectrum quadratic, MLP) are in the table only
UNWRAP_ONLY = tuple(m for m in RESULTS_B if not m.endswith("+ unwrap"))
show_results(RESULTS_B, f"Part B ({LANTERN}): 41 wavelengths, +-{B_SPAN_NM / 1000:g} um, each method + unwrap",
             dof=DOF_B, sort=True, logy=True, exclude=UNWRAP_ONLY)

## B6. Part B: what differs between lanterns

- *pl19_800:* the per-wavelength OMP models + unwrap give the best result
  (1.64 nm); the full-spectrum MLP reaches 10.25 nm.
- *pl7_1200:* the per-wavelength route fails (1688 nm, 52% wrong fringes):
  each per-wavelength model has to recover pistons wrapped over a full
  +-lambda/2, the single-wavelength regime where 7 signals are ambiguous
  (Part A), so its wrapped error (~200 nm, against ~350 nm for chance) is far
  too large for the unwrap. Only the full-spectrum MLP learns much (258 nm
  against 2893 nm for no skill): it sees all wavelengths at once, enough
  signals to resolve what no single wavelength can.
- *pl19_1200:* see its executed copy in `runs/pl19_1200/`.


# Summary

Every model of Parts A and B. The parts are not directly comparable:
different ranges (+-lambda/2 vs +-8 lambda), signals (camera at the design wavelength
with unit coupled power vs. channel power fractions at 41 wavelengths) and
training-set sizes.


In [ ]:
print(f"{'':36s} {'mean RMSE':>10s}   per piston [nm]")
for part, res in [(f"A  {LANTERN}, one wavelength, +-{A_SPAN_NM:.0f} nm", RESULTS_A),
                  (f"B  {LANTERN}, 41 wl, +-{B_SPAN_NM / 1000:g} um", RESULTS_B)]:
    print(part)
    for m, r in sorted(res.items(), key=lambda kv: kv[1].mean()):
        print(f"  {m:34s} {r.mean():10.2f}   {np.round(r, 2)}")
